# Notebook 03: Kiểm tra Pipeline và Chống Rò rỉ Dữ liệu

Notebook này thực hiện các kiểm tra C1–C10 theo kế hoạch triển khai TV3.
**Tất cả kiểm tra phải đạt (PASS) trước khi chạy thí nghiệm chính.**

Lưu ý: Notebook này chỉ chứa mã kiểm tra ngắn, không chứa logic huấn luyện chính.

In [4]:
import sys
import os
import numpy as np
import pandas as pd

# Thêm src/ vào sys.path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if 'notebooks' in os.getcwd() else 'src')
# Thử cả ../src
sys.path.insert(0, '../src')

from preprocessing import (
    RAW_PATH, CLEANED_PATH, FEATURE_COLUMNS, TARGET_COLUMN,
    clean_raw_data, get_feature_target_split
)
from config import (
    SEED, N_SPLITS, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS,
    EXCLUDED_COLUMNS, FEATURE_COLUMNS as FC
)
from pipelines import build_preprocessor, get_models
from train import load_xy, make_folds, run_cv_experiment
from evaluation import compute_fold_metrics, get_scores
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

print('Import thành công')
print(f'FEATURE_COLUMNS: {FEATURE_COLUMNS}')
print(f'TARGET_COLUMN: {TARGET_COLUMN}')

Import thành công
FEATURE_COLUMNS: ['Choose your gender', 'Age', 'What is your course?', 'Your current year of Study', 'What is your CGPA?', 'Marital status']
TARGET_COLUMN: Do you have Depression?


In [2]:
# C1: Hình dạng dữ liệu
X, y = load_xy()
assert X.shape == (101, 6), f'[FAIL C1] X.shape = {X.shape}, kỳ vọng (101, 6)'
vc = y.value_counts()
assert vc[0] == 66 and vc[1] == 35, f'[FAIL C1] phân bố y = {dict(vc)}, kỳ vọng {{0:66, 1:35}}'
assert X['Age'].isna().sum() == 1, f'[FAIL C1] Age NaN count = {X["Age"].isna().sum()}, kỳ vọng 1'
print('✅ C1 PASS — X.shape=(101,6), y={0:66,1:35}, Age có 1 NaN')

[load_xy] X.shape = (101, 6), y.value_counts() =
Do you have Depression?
0    66
1    35
✅ C1 PASS — X.shape=(101,6), y={0:66,1:35}, Age có 1 NaN


In [3]:
# C2: Cột
assert list(X.columns) == FEATURE_COLUMNS, f'[FAIL C2] X.columns = {list(X.columns)}'
for col in EXCLUDED_COLUMNS:
    assert col not in X.columns, f'[FAIL C2] Cột bị loại "{col}" vẫn có trong X'
print('✅ C2 PASS — Đúng 6 cột, 4 cột bị loại không có trong X')
print(f'   Cột X: {list(X.columns)}')

✅ C2 PASS — Đúng 6 cột, 4 cột bị loại không có trong X
   Cột X: ['Choose your gender', 'Age', 'What is your course?', 'Your current year of Study', 'What is your CGPA?', 'Marital status']


In [4]:
# C3: Khớp TV1
raw = pd.read_csv(RAW_PATH, encoding='utf-8')
cleaned_code = clean_raw_data(raw)
cleaned_file = pd.read_csv(CLEANED_PATH, encoding='utf-8')

for col in FEATURE_COLUMNS + [TARGET_COLUMN]:
    code_vals = cleaned_code[col].reset_index(drop=True)
    file_vals = cleaned_file[col].reset_index(drop=True)
    # So sánh với NaN-safe
    equal = code_vals.equals(file_vals)
    assert equal, f'[FAIL C3] Cột "{col}" không khớp giữa clean_raw_data() và file CSV'
print('✅ C3 PASS — clean_raw_data() khớp với student_mental_health_cleaned.csv')

✅ C3 PASS — clean_raw_data() khớp với student_mental_health_cleaned.csv


In [5]:
# C4: Fold
folds = make_folds(X, y)
expected_test_sizes = {21, 20, 20, 20, 20}
test_sizes = [len(test) for _, test in folds]
assert set(test_sizes) == expected_test_sizes or sorted(test_sizes) == sorted([21,20,20,20,20]), \
    f'[FAIL C4] test_sizes = {test_sizes}'

for fold_idx, (_, test) in enumerate(folds):
    n_yes = int(y.iloc[test].sum())
    assert n_yes == 7, f'[FAIL C4] Fold {fold_idx}: n_yes = {n_yes}, kỳ vọng 7'

# Mọi chỉ số dòng xuất hiện đúng 1 lần trong test
all_test_indices = []
for _, test in folds:
    all_test_indices.extend(test.tolist())
assert len(all_test_indices) == 101, f'[FAIL C4] Tổng test_idx = {len(all_test_indices)}, kỳ vọng 101'
assert len(set(all_test_indices)) == 101, '[FAIL C4] Có chỉ số dòng trùng lặp trong test folds'
print('✅ C4 PASS — test_sizes =', test_sizes, ', mỗi fold 7 ca Yes, mọi index xuất hiện 1 lần')

[make_folds] Đã lưu D:\Documents\HOC TAP (D)\TRI TUE NHAN TAO\Mental-Student-Health-\results\ml\tables\fold_assignments.csv
  Fold 0: test_size=21, n_yes=7
  Fold 1: test_size=20, n_yes=7
  Fold 2: test_size=20, n_yes=7
  Fold 3: test_size=20, n_yes=7
  Fold 4: test_size=20, n_yes=7
✅ C4 PASS — test_sizes = [21, 20, 20, 20, 20] , mỗi fold 7 ca Yes, mọi index xuất hiện 1 lần


In [6]:
# C5: Imputer không rò rỉ
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)
global_age_median = X_reset['Age'].median()

for fold_idx, (train_idx, test_idx) in enumerate(folds):
    X_train = X_reset.iloc[train_idx]
    prep = build_preprocessor(scale=False)
    prep.fit(X_train, y_reset.iloc[train_idx])
    # Lấy imputer của cột Age (transformer 'num', bước 'imputer')
    num_transformer = prep.named_transformers_['num']
    imputed_median = num_transformer.named_steps['imputer'].statistics_[0]
    train_age_median = X_train['Age'].median()
    assert abs(imputed_median - train_age_median) < 1e-9, \
        f'[FAIL C5] Fold {fold_idx}: imputer median={imputed_median}, train_median={train_age_median}'
    print(f'  Fold {fold_idx}: imputer_median={imputed_median}, train_median={train_age_median}, '
          f'global_median={global_age_median} (khác={abs(imputed_median-global_age_median):.2f})')

print('✅ C5 PASS — Imputer fit đúng trên train fold, không rò rỉ từ test')

  Fold 0: imputer_median=19.0, train_median=19.0, global_median=19.0 (khác=0.00)
  Fold 1: imputer_median=19.0, train_median=19.0, global_median=19.0 (khác=0.00)
  Fold 2: imputer_median=19.0, train_median=19.0, global_median=19.0 (khác=0.00)


  Fold 3: imputer_median=19.0, train_median=19.0, global_median=19.0 (khác=0.00)
  Fold 4: imputer_median=19.0, train_median=19.0, global_median=19.0 (khác=0.00)
✅ C5 PASS — Imputer fit đúng trên train fold, không rò rỉ từ test


In [7]:
# C6: Không fit ngoài pipeline
import subprocess
result = subprocess.run(
    ['python', '-c', 
     'import ast, sys; src=["src/config.py","src/pipelines.py","src/evaluation.py","src/train.py"]; '
     '[print(f"{f}:{i+1}: {line.rstrip()}") for f in src for i,line in enumerate(open(f,encoding="utf-8")) '
     'if "fit(" in line and "def " not in line and "#" not in line.lstrip()[:2] '
     'and "clone" not in line and "GridSearch" not in line and "pipeline" not in line.lower() '
     'and "pipe" not in line.lower() and "fitted" not in line and "get_scores" not in line '
     'and "_fit" not in line]'
    ],
    capture_output=True, text=True, cwd='..'
)
print('Kết quả grep fit trong src/*.py (kiểm tra thủ công):')
# In tất cả dòng có fit để kiểm tra thủ công
for f in ['../src/config.py', '../src/pipelines.py', '../src/evaluation.py', '../src/train.py']:
    with open(f, encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            if '.fit(' in line and 'def ' not in line:
                print(f'  {f}:{i+1}: {line.rstrip()}')

print('\n✅ C6 — Kiểm tra thủ công: mọi .fit() ở trên đều trong pipeline hoặc clone()')

Kết quả grep fit trong src/*.py (kiểm tra thủ công):
  ../src/evaluation.py:765:             pipe.fit(X_train, y_train)
  ../src/evaluation.py:807:     pipe.fit(X, y)
  ../src/train.py:175:             pipe.fit(X_train, y_train)
  ../src/train.py:280:             gs.fit(X_train, y_train)
  ../src/train.py:314:         dummy_pipe.fit(X_train, y_train)
  ../src/train.py:388:             pipe.fit(X_train, y_train)

✅ C6 — Kiểm tra thủ công: mọi .fit() ở trên đều trong pipeline hoặc clone()


In [8]:
# C7: Nhãn xáo trộn (permutation test)
from sklearn.metrics import roc_auc_score

models = get_models()
models_to_test = {'LogisticRegression': models['LogisticRegression'],
                  'RandomForest': models['RandomForest']}

rng = np.random.default_rng(seed=SEED)
n_perm = 20

for model_name, pipeline in models_to_test.items():
    auc_scores = []
    for perm_idx in range(n_perm):
        y_perm = pd.Series(rng.permutation(y.values), index=y.index)
        m_df, _ = run_cv_experiment(
            f'permtest_{model_name}_{perm_idx}',
            {model_name: pipeline},
            X, y_perm, folds,
            save=False
        )
        avg_auc = m_df['roc_auc'].mean()
        auc_scores.append(avg_auc)
    
    mean_auc = np.mean(auc_scores)
    print(f'{model_name}: mean ROC-AUC trên {n_perm} hoán vị = {mean_auc:.3f}')
    assert 0.40 <= mean_auc <= 0.60, \
        f'[FAIL C7] {model_name}: mean_auc={mean_auc:.3f} ngoài [0.40, 0.60]'

print('✅ C7 PASS — ROC-AUC trung bình với nhãn xáo trộn ≈ 0.5 (không rò rỉ)')

  [permtest_LogisticRegression_0] Mô hình: LogisticRegression


  [permtest_LogisticRegression_1] Mô hình: LogisticRegression
  [permtest_LogisticRegression_2] Mô hình: LogisticRegression


  [permtest_LogisticRegression_3] Mô hình: LogisticRegression
  [permtest_LogisticRegression_4] Mô hình: LogisticRegression


  [permtest_LogisticRegression_5] Mô hình: LogisticRegression


  [permtest_LogisticRegression_6] Mô hình: LogisticRegression


  [permtest_LogisticRegression_7] Mô hình: LogisticRegression


... (26 outputs left out)

  [permtest_RandomForest_17] Mô hình: RandomForest


  [permtest_RandomForest_18] Mô hình: RandomForest


  [permtest_RandomForest_19] Mô hình: RandomForest


RandomForest: mean ROC-AUC trên 20 hoán vị = 0.505
✅ C7 PASS — ROC-AUC trung bình với nhãn xáo trộn ≈ 0.5 (không rò rỉ)


In [9]:
# C8: Xác định (chạy E0 hai lần, kết quả y hệt)
models_e0 = get_models()

m1, _ = run_cv_experiment('E0_check1', models_e0, X, y, folds, save=False)
m2, _ = run_cv_experiment('E0_check2', models_e0, X, y, folds, save=False)

m1_vals = m1[['model','fold','f1_yes','roc_auc']].sort_values(['model','fold']).reset_index(drop=True)
m2_vals = m2[['model','fold','f1_yes','roc_auc']].sort_values(['model','fold']).reset_index(drop=True)

pd.testing.assert_frame_equal(m1_vals, m2_vals, check_exact=False, rtol=1e-10)
print('✅ C8 PASS — Chạy E0 hai lần cho kết quả y hệt (xác định)')

  [E0_check1] Mô hình: Dummy
  [E0_check1] Mô hình: LogisticRegression


  [E0_check1] Mô hình: DecisionTree
  [E0_check1] Mô hình: RandomForest


  [E0_check1] Mô hình: SVM


  [E0_check2] Mô hình: Dummy
  [E0_check2] Mô hình: LogisticRegression


  [E0_check2] Mô hình: DecisionTree
  [E0_check2] Mô hình: RandomForest


  [E0_check2] Mô hình: SVM


✅ C8 PASS — Chạy E0 hai lần cho kết quả y hệt (xác định)


In [10]:
# C9: Dummy
import pandas as pd
metrics_df = pd.read_csv('../results/ml/tables/metrics_per_fold.csv', encoding='utf-8')
dummy = metrics_df[(metrics_df['experiment']=='E0_main') & (metrics_df['model']=='Dummy')]

for _, row in dummy.iterrows():
    assert row['recall_yes'] == 0.0, f'[FAIL C9] Fold {row["fold"]}: recall = {row["recall_yes"]}'
    assert row['f1_yes'] == 0.0, f'[FAIL C9] Fold {row["fold"]}: f1 = {row["f1_yes"]}'
    assert abs(row['roc_auc'] - 0.5) < 1e-9, f'[FAIL C9] Fold {row["fold"]}: roc_auc = {row["roc_auc"]}'

mean_acc = dummy['accuracy'].mean()
print(f'Dummy accuracy trung bình: {mean_acc:.4f} (kỳ vọng ≈ 0.653)')
assert 0.63 <= mean_acc <= 0.68, f'[FAIL C9] accuracy trung bình = {mean_acc}'

mean_pr_auc = dummy['pr_auc'].mean()
print(f'Dummy PR-AUC trung bình: {mean_pr_auc:.4f} (kỳ vọng ≈ 0.333–0.350)')
assert 0.32 <= mean_pr_auc <= 0.36, f'[FAIL C9] PR-AUC trung bình = {mean_pr_auc}'

print('✅ C9 PASS — Dummy: recall=0, f1=0, roc_auc=0.5, accuracy≈0.653')

Dummy accuracy trung bình: 0.6533 (kỳ vọng ≈ 0.653)
Dummy PR-AUC trung bình: 0.3467 (kỳ vọng ≈ 0.333–0.350)
✅ C9 PASS — Dummy: recall=0, f1=0, roc_auc=0.5, accuracy≈0.653


In [11]:
# C10: Độ rộng one-hot
print('Số cột sau one-hot (cấu hình onehot) ở mỗi fold:')
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

for fold_idx, (train_idx, test_idx) in enumerate(folds):
    X_train = X_reset.iloc[train_idx]
    prep = build_preprocessor(scale=False, course_mode='onehot')
    prep.fit(X_train, y_reset.iloc[train_idx])
    n_cols = prep.transform(X_train).shape[1]
    feature_names = prep.get_feature_names_out()
    print(f'  Fold {fold_idx}: {n_cols} cột sau one-hot')

print(f'\n  (Tên cột: {list(feature_names[:5])} ... {list(feature_names[-3:])})')
print('✅ C10 — Thông tin: số cột thay đổi theo fold do Course có nhiều hạng mục hiếm')

Số cột sau one-hot (cấu hình onehot) ở mỗi fold:
  Fold 0: 53 cột sau one-hot
  Fold 1: 57 cột sau one-hot
  Fold 2: 54 cột sau one-hot
  Fold 3: 53 cột sau one-hot
  Fold 4: 53 cột sau one-hot

  (Tên cột: ['Age', 'Choose your gender_Female', 'Choose your gender_Male', 'Your current year of Study_year 1', 'Your current year of Study_year 2'] ... ['What is your course?_Usuluddin', 'What is your course?_koe', 'What is your course?_psychology'])
✅ C10 — Thông tin: số cột thay đổi theo fold do Course có nhiều hạng mục hiếm


In [12]:
# Tổng kết
print('=' * 60)
print('TỔNG KẾT KIỂM TRA CHỐNG RÒ RỈ VÀ SANITY CHECK')
print('=' * 60)
checks = [
    'C1: Hình dạng dữ liệu — PASS',
    'C2: Cột feature và cột bị loại — PASS',
    'C3: Khớp TV1 (clean_raw_data vs file CSV) — PASS',
    'C4: Fold (kích thước, tỷ lệ nhãn, không trùng) — PASS',
    'C5: Imputer fit trên train fold (không rò rỉ) — PASS',
    'C6: Kiểm tra thủ công fit() — PASS (xem ô trên)',
    'C7: Nhãn xáo trộn → ROC-AUC ≈ 0.5 — PASS',
    'C8: Xác định (chạy 2 lần → kết quả giống nhau) — PASS',
    'C9: Dummy baseline — PASS',
    'C10: Thông tin độ rộng one-hot theo fold — PASS (thông tin)',
]
for c in checks:
    print(f'  ✅ {c}')

TỔNG KẾT KIỂM TRA CHỐNG RÒ RỈ VÀ SANITY CHECK
  ✅ C1: Hình dạng dữ liệu — PASS
  ✅ C2: Cột feature và cột bị loại — PASS
  ✅ C3: Khớp TV1 (clean_raw_data vs file CSV) — PASS
  ✅ C4: Fold (kích thước, tỷ lệ nhãn, không trùng) — PASS
  ✅ C5: Imputer fit trên train fold (không rò rỉ) — PASS
  ✅ C6: Kiểm tra thủ công fit() — PASS (xem ô trên)
  ✅ C7: Nhãn xáo trộn → ROC-AUC ≈ 0.5 — PASS
  ✅ C8: Xác định (chạy 2 lần → kết quả giống nhau) — PASS
  ✅ C9: Dummy baseline — PASS
  ✅ C10: Thông tin độ rộng one-hot theo fold — PASS (thông tin)
